In [1]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import pygame

In [2]:
environment_name = "CartPole-v1"
env = gym.make(environment_name, render_mode="human")

In [8]:
environment_name = "CartPole-v1"
env = gym.make(environment_name, render_mode="human")

episodes = 5
for episode in range(1, episodes + 1):
    state, info = env.reset()
    done = False
    truncated = False
    score = 0
    while not done and not truncated:
        env.render()
        action = env.action_space.sample()
        n_state, reward, done, truncated, info = env.step(action)
        score += reward
    print("Episode: {} Score: {}".format(episode, score))

Episode: 1 Score: 20.0
Episode: 2 Score: 43.0
Episode: 3 Score: 25.0
Episode: 4 Score: 13.0
Episode: 5 Score: 38.0


In [ ]:
env.reset()

(array([0.00478037, 0.03467751, 0.01833696, 0.04663046], dtype=float32), {})

In [11]:
env.action_space

Discrete(2)

In [17]:
env.action_space.sample()

np.int64(0)

In [19]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [18]:
env.observation_space.sample()

array([ 2.2745245 , -0.33638927, -0.4070809 ,  0.5838357 ], dtype=float32)

In [21]:
log_path = os.path.join("Training", "Logs")

In [22]:
log_path

'Training/Logs'

In [36]:
env = gym.make(environment_name)
env = DummyVecEnv([lambda: env])
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log = log_path)

Using cpu device


In [37]:
model.learn(total_timesteps=20000)

Logging to Training/Logs/PPO_4
------------------------------
| time/              |       |
|    fps             | 10061 |
|    iterations      | 1     |
|    time_elapsed    | 0     |
|    total_timesteps | 2048  |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 6810        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008188503 |
|    clip_fraction        | 0.0822      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.687      |
|    explained_variance   | -0.0278     |
|    learning_rate        | 0.0003      |
|    loss                 | 6.26        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0129     |
|    value_loss           | 52          |
--------------------------------------

In [27]:
PPO_Path = os.path.join("Training", "Saved Models", "PPO_Model_Cartpole")

In [94]:
model.save(PPO_Path)

In [29]:
del model

In [31]:
model = PPO.load(PPO_Path)

In [47]:
env_eval = gym.make(environment_name, render_mode="human")
env_eval = DummyVecEnv([lambda: env_eval])

In [48]:

evaluate_policy(model, env_eval, n_eval_episodes=10)

/Users/elvincheung/PersonalProjects/nba_rl/.venv/lib/python3.14/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(500.0), np.float64(0.0))

In [70]:
episodes = 5
for episode in range(1, episodes + 1):
    obs = env.reset()
    done = False
    score = 0
    while not done:
        env.render()
        action, _ = model.predict(obs)
        obs, reward, done, info = env.step(action)
        score += reward
    print("Episode: {} Score: {}".format(episode, score))

Episode: 1 Score: [500.]
Episode: 2 Score: [500.]
Episode: 3 Score: [500.]
Episode: 4 Score: [495.]
Episode: 5 Score: [387.]


In [92]:
env.reset()

array([[0.02144039, 0.01129809, 0.02865497, 0.02477065]], dtype=float32)

In [95]:
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

In [97]:
save_path = os.path.join("Training", "Saved Models")

In [112]:
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=500, verbose=1)
eval_callback = EvalCallback(env, 
callback_on_new_best=stop_callback,
eval_freq=1000,
best_model_save_path=save_path,
verbose=1)

In [102]:
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device


In [103]:
model.learn(total_timesteps=40000, callback=eval_callback)

Logging to Training/Logs/PPO_6
Eval num_timesteps=1000, episode_reward=58.60 +/- 17.00
Episode length: 58.60 +/- 17.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 58.6     |
|    mean_reward     | 58.6     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Eval num_timesteps=2000, episode_reward=51.40 +/- 11.74
Episode length: 51.40 +/- 11.74
---------------------------------
| eval/              |          |
|    mean_ep_length  | 51.4     |
|    mean_reward     | 51.4     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
-----------------------------
| time/              |      |
|    fps             | 8055 |
|    iterations      | 1    |
|    time_elapsed    | 0    |
|    total_timesteps | 2048 |
-----------------------------
Eval num_timesteps=3000, episode_reward=138.20 +/- 68.61
Episode length: 138.20 +/- 6

In [ ]:
net_arch = (dict(pi=[128,128,128,128], vf=[128,128,128,128]))

In [110]:
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path, policy_kwargs={"net_arch":net_arch})

Using cpu device


In [111]:
model.learn(total_timesteps=40000, callback=eval_callback)

Logging to Training/Logs/PPO_7
Eval num_timesteps=1000, episode_reward=31.60 +/- 7.68
Episode length: 31.60 +/- 7.68
---------------------------------
| eval/              |          |
|    mean_ep_length  | 31.6     |
|    mean_reward     | 31.6     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
Eval num_timesteps=2000, episode_reward=42.20 +/- 18.33
Episode length: 42.20 +/- 18.33
---------------------------------
| eval/              |          |
|    mean_ep_length  | 42.2     |
|    mean_reward     | 42.2     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
-----------------------------
| time/              |      |
|    fps             | 6229 |
|    iterations      | 1    |
|    time_elapsed    | 0    |
|    total_timesteps | 2048 |
-----------------------------


/Users/elvincheung/PersonalProjects/nba_rl/.venv/lib/python3.14/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=3000, episode_reward=209.80 +/- 153.84
Episode length: 209.80 +/- 153.84
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 210         |
|    mean_reward          | 210         |
| time/                   |             |
|    total_timesteps      | 3000        |
| train/                  |             |
|    approx_kl            | 0.015117911 |
|    clip_fraction        | 0.222       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.681      |
|    explained_variance   | -0.00267    |
|    learning_rate        | 0.0003      |
|    loss                 | 3.16        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0227     |
|    value_loss           | 19.1        |
-----------------------------------------
Eval num_timesteps=4000, episode_reward=92.80 +/- 39.37
Episode length: 92.80 +/- 39.37
---------------------------------
| eval/              |        

In [114]:
from stable_baselines3 import DQN

In [115]:
model = DQN("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device


In [ ]:
model.learn(total_timesteps=40000)

Logging to Training/Logs/DQN_1
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.973    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 4114     |
|    time_elapsed     | 0        |
|    total_timesteps  | 113      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.597    |
|    n_updates        | 3        |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.945    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 4996     |
|    time_elapsed     | 0        |
|    total_timesteps  | 233      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.568    |
|    n_updates        | 33       |
----------------------------------
----------------------------------
| rollout/            | 

: 